# LangChain

- A tool that helps in invoking a wide variety of AI models and manage them in end - end applications. (Orchestration)
- can help connecting the agent to system DB or other data sources. (RAGs)

A framework that makes it easy to build AI Apps.

## Topic:

- Langchain basic
- Prompt templates
  - chains
  - chat roles with prompt templates
- Agent using langchain
  - tool calls
  - agentic loop
- langchain middlewares
- Providers
- Short Term Memory

## Imports and Dependencies

In [1]:
import re
import json
from pathlib import Path
from fnmatch import fnmatch
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv
from enum import Enum
from typing import List, Optional

# Lang chain
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()
model = "openrouter/free"

llm = ChatOpenRouter(
    model_name=model,
)

## Basics:

In [2]:
prompt = "what is binary search. Write code for it."

response = llm.invoke([
    HumanMessage(content=prompt)
])

print(response.content)

### What is Binary Search?

**Binary Search** is an efficient algorithm used to find the position of a target value within a **sorted array**. 

Unlike a linear search (which checks every single element one by one), binary search uses a "divide and conquer" strategy. It repeatedly divides the search interval in half.

#### How it works:
1.  **Start** with the entire sorted array.
2.  **Find the middle element.**
3.  **Compare** the middle element with your target value:
    *   If the target **equals** the middle element, you found it! Return the index.
    *   If the target is **smaller** than the middle element, discard the right half and repeat the search on the left half.
    *   If the target is **larger** than the middle element, discard the left half and repeat the search on the right half.
4.  **Repeat** until the target is found or the search interval is empty.

#### Efficiency (Complexity):
*   **Time Complexity:** $O(\log n)$ — This is extremely fast. For example, searching 

- LangChain provides a different API than the ones present from OpenAI client.
- For Passing Human and Assistant message inbuilt functions already exist.
- The concepts for agentic loop etc are the same here as well.
- https://docs.langchain.com/oss/python/integrations/chat/openrouter - for other parameters when creating the model.
  - for tools and stuff the lang chain follows chat completions api format, check above docs
  - other paramerters like `max_tokens` also exist to limit the output tokens.

## Prompt Template and Chains:

- Reusable instructions which can be passed to langchain with different parameters.

In [3]:
sample_template = """
You are an expert CS Engineer explain the following algorithm :-
{algorithm_name}

Follow the exact structure :-

### Concept
Explain what problem this algo solves.

### Efficiency
Explain time and space complexity

### Sample Trace run
Show a simple dry run using {sample_input}

Answer::
"""

sample_prompt_template = PromptTemplate(template=sample_template, input_variables=["algorithm_name", "sample_input"])

In [4]:
prompt = sample_prompt_template.format(algorithm_name="Binary Search", sample_input="1 2 3 4 5")
print(prompt)
print("\n*****\n")
print(llm.invoke(prompt).content)
print("\n*****\n")


You are an expert CS Engineer explain the following algorithm :-
Binary Search

Follow the exact structure :-

### Concept
Explain what problem this algo solves.

### Efficiency
Explain time and space complexity

### Sample Trace run
Show a simple dry run using 1 2 3 4 5

Answer::


*****

### Concept
Binary Search is a classic algorithm for finding a target value within a **sorted** array (or list).  
The problem it solves is: *given a sorted collection of items and a key to locate, determine the index of that key (or confirm its absence) in the smallest number of comparisons possible*.  
Instead of scanning every element (linear search), binary search repeatedly halves the search interval, discarding the half that cannot contain the target. This yields logarithmic time complexity.

### Efficiency
- **Time Complexity:**  
  - Best case: O(1) – the middle element is the target on the first check.  
  - Average and worst case: O(log n) – the search space is halved each iteration, where

---
- Above was one way we could invoke our new prompt.
- However langchain allows us to chain operations like invoking llm generating prompts

---

#### Chains:

In [5]:
# prompt feeded inside the LLM
sample_chain = sample_prompt_template | llm
result = sample_chain.invoke({
    "algorithm_name":"Binary Search", 
    "sample_input":"1 2 3 4 5"
})
print(result.content)

### Concept
Binary Search is an efficient algorithm for finding a target value within a **sorted** array or list. It works by repeatedly dividing the search interval in half. If the value of the search key is less than the item in the middle of the interval, the algorithm narrows the search to the lower half. Otherwise, it narrows the search to the upper half. This process continues until the target is found or the interval is empty. It is significantly faster than linear search for large datasets.

### Efficiency
*   **Time Complexity:** $O(\log n)$. In the worst case, the algorithm performs a comparison with half of the remaining elements at each step. This results in a logarithmic number of steps relative to the input size $n$.
*   **Space Complexity:** $O(1)$ for the iterative implementation (using constant extra space for pointers). If implemented recursively, the space complexity is $O(\log n)$ due to the recursion stack depth.

### Sample Trace run
Consider a sorted array `[1, 2

---
- Above we created a simple chain and passed our parameter directly in chain in a dictionary which would be fed to prompt template which is then fed into llm
---

#### Chat Roles:

In [6]:
sample_template_2 = ChatPromptTemplate.from_messages([
    {"role": "system", "content":"some system prompt"},
    {"role":"human", "content": "some human prompt"},
    {"role": "ai","content": "some ai response"}
])

- Now we can create chain with this and invoke the llm.
- Everything remains the same as before.

## Agents in Lang Chain:
- Previously we manually handled writing agentic loop
  - handling tool call
  - passing messages
  - max turn tracking etc
- Lang chain simplifies this process

https://docs.langchain.com/oss/python/langchain/agents

- The `create_agent` allows you to create a complete harness not just an agentic loop.
  - it automatically converts the messages into appropriate messages with roles.
  - you can pass tools information.
    - It automatically handles tool schema definition.
  - mcp information.
  - skills.
  - memory.
  - middlewares.

In [7]:
@tool
def get_weather(location: str) -> str:
    """
    This returns the weather of the provided location
    """
    return "The weather is 20 Celcius"

In [8]:
llm = ChatOpenRouter(
    model_name=model,
)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a weather agent. You are given a location and are asked to return the weather of that location."
)

In [9]:
result = agent.invoke({
    "messages":[
        HumanMessage(content="Weather in NCR")
    ]
})
print(result)

{'messages': [HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='84d964fe-b743-4294-b057-bece8bcacf76'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the weather in "NCR". This likely refers to the National Capital Region (NCR), which could be Delhi NCR (National Capital Region of India) or possibly New York City (NCR could stand for New York City). However, given the context of weather queries, it\'s most commonly referring to the National Capital Region of India (Delhi, Mumbai, etc.).\n\nLet me use the get_weather function to fetch the weather for "NCR". I\'ll need to specify the location parameter. Since "NCR" is a common abbreviation for the National Capital Region in India, I\'ll try that first. If it doesn\'t return useful results, I might need to clarify or try a different interpretation.\n\nActually, looking at the function, it requires a "location" parameter as a string. I should try "NCR" as the loca

#### Analyzing the output:

**Complete Response:**

```python
{'messages': [HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='ae3b5bda-b573-4f71-abd8-9694ed0aa940'), AIMessage(content="I'll get the weather for you! The NCR (National Capital Region) typically includes areas like Delhi, Gurugram, Noida, and Faridabad.\n\n", additional_kwargs={'reasoning_content': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.'}]}, response_metadata={'model_name': 'recursive/flint-v1.0:free', 'id': 'gen-1794312420-Tzld41jT98hXD2nNsDE4', 'created': 1794312420, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-bc77-73f8-98e4-7897a88231e6-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NCR'}, 'id': 'call_0b3b92b8cf4840b9aae2eead', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 150, 'total_tokens': 306, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 124}}), ToolMessage(content='The weather is 20 Celcius', name='get_weather', id='36a3050f-e189-41b7-b4d1-352a0b99ed07', tool_call_id='call_0b3b92b8cf4840b9aae2eead'), AIMessage(content='The current weather in NCR (National Capital Region) is **20°C**.', additional_kwargs={'reasoning_content': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'id': 'gen-1794312422-7SQKVbFKV7m3qByLYbRI', 'created': 1794312422, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-c4a7-75df-8091-98de24e777c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 496, 'output_tokens': 45, 'total_tokens': 541, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 26}})]}
```

**result["messages"]**

```python
[HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='ae3b5bda-b573-4f71-abd8-9694ed0aa940'),
 
 AIMessage(content="I'll get the weather for you! The NCR (National Capital Region) typically includes areas like Delhi, Gurugram, Noida, and Faridabad.\n\n", additional_kwargs={'reasoning_content': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.'}]}, response_metadata={'model_name': 'recursive/flint-v1.0:free', 'id': 'gen-1794312420-Tzld41jT98hXD2nNsDE4', 'created': 1794312420, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-bc77-73f8-98e4-7897a88231e6-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NCR'}, 'id': 'call_0b3b92b8cf4840b9aae2eead', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 150, 'total_tokens': 306, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 124}}),
 
 ToolMessage(content='The weather is 20 Celcius', name='get_weather', id='36a3050f-e189-41b7-b4d1-352a0b99ed07', tool_call_id='call_0b3b92b8cf4840b9aae2eead'),
 
 AIMessage(content='The current weather in NCR (National Capital Region) is **20°C**.', additional_kwargs={'reasoning_content': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'id': 'gen-1794312422-7SQKVbFKV7m3qByLYbRI', 'created': 1794312422, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-c4a7-75df-8091-98de24e777c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 496, 'output_tokens': 45, 'total_tokens': 541, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 26}})]
```
- We see the `HumanMessage` - we sent this.
- We see the `AIMessage` - AI reasoning output. 
  - We see that a tool call is requested as `tool_calls` is not empty.
- `ToolMessage` object denotes that the tool call was made and the output is returned.
  - the `tool_call_id` is same as in previous message to denote that this tool response is for this requested tool call.
- Another `AIMessage` denoting final answer.

**Lang chain has abstracted away the whole process of agentic loop and tool calling, the only thing we had to do was create the agent and provide tools, system prompt and messages**

In [10]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Weather in NCR
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7a20ad3cf5194303b73d5aa1)
 Call ID: call_7a20ad3cf5194303b73d5aa1
  Args:
    location: NCR
================================= Tool Message =================================
Name: get_weather

The weather is 20 Celcius
================================== Ai Message ==================================



The weather in NCR is 20°C.


### Files Agent:

#### Constants:

In [11]:
PROJECT_ROOT = Path.cwd() / "tmp"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
print(PROJECT_ROOT)

/Users/prakharkumar/Desktop/projects/AI-Engineering/tmp


In [12]:
MAX_MODEL_CALLS_PER_RUN = "10"
MAX_READ_BYTES = "100000000"
AGENT_NAME = "Files Agent"

#### Utilities:

In [13]:
def hitl_enabled() -> bool:
    return os.getenv("HITL_ENABLED", "true").lower() in {"1", "true", "yes"}

In [14]:
BLOCKED_PATH_PATTERN = [
    ".env",
    ".env.*",
    ".git*",
    "*.log*",
    ".key",
    ".pem",
    ".secret",
]

In [15]:
def normalize_path(path: str) -> str:
    normalized = Path.as_posix(path)
    if normalized.startswith("./"):
        normalized = normalized[2:]

    return normalized

In [16]:
def is_blocked_path(path: str) -> str:
    normalized = normalize_path(path)
    return any(fnmatch(normalized, pattern) for pattern in BLOCKED_PATH_PATTERN)

In [17]:
# if agent tries to go outside the workdir path raise an error
def resolve_work_path(path: str) -> Path:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    candidate = (PROJECT_ROOT / path).resolve()
    try:
        candidate.relative_to(PROJECT_ROOT)
    except ValueError as e:
        raise ValueError(f"Path escapes working directory")
    return candidate

#### Tools

In [18]:
@tool
def read_file(path: str) -> str:
    """Reads a UTF-8 file from the working directory.

    Args:
        path: Relative path of the file.
    """
    file_path = resolve_work_path(path)
    try:
        return file_path.read_text(encoding="utf-8")
    except FileNotFoundError:
        raise FileNotFoundError("File does not exist")
    except PermissionError:
        raise PermissionError("Permission Error")
    except Exception as e:
        raise Exception(f"Error reading the file: {e}")

In [19]:
@tool
def list_files(path: str = "") -> List[str]:
    """Lists files and directories located inside the target path within the workspace.

    Args:
        path: Relative path of the directory. Defaults to the workspace root.

    Returns:
        A list of relative path strings representing the directory contents.
    """
    target_dir = resolve_work_path(path)
    try:
        if not target_dir.exists():
            raise FileNotFoundError("Directory does not exist")
        if not target_dir.is_dir():
            raise NotADirectoryError(f"'{path}' is not a directory")

        # Returns paths relative to the queried directory
        return [str(item.relative_to(target_dir)) for item in target_dir.iterdir()]
    except FileNotFoundError:
        raise FileNotFoundError("Directory does not exist")
    except NotADirectoryError as e:
        raise NotADirectoryError(str(e))
    except PermissionError:
        raise PermissionError("Permission Error")
    except Exception as e:
        raise Exception(f"Error listing directory: {e}")

In [20]:
@tool
def write_file(path: str, content: str, overwrite: bool = True) -> str:
    """Writes UTF-8 text content to a file in the working directory.

    Creates parent directories automatically if they do not exist.

    Args:
        path: Relative path of the file to write.
        content: The text content to write.
        overwrite: If False, raises an error if the file already exists.

    Returns:
        A confirmation message indicating success.
    """
    file_path = resolve_work_path(path)
    try:
        if not overwrite and file_path.exists():
            raise FileExistsError(f"File '{path}' already exists and overwrite is set to False")

        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return f"Successfully wrote to {path}"
    except FileExistsError as e:
        raise FileExistsError(str(e))
    except PermissionError:
        raise PermissionError("Permission Error")
    except Exception as e:
        raise Exception(f"Error writing to file: {e}")

In [21]:
@tool
def edit_file(
    path: str, target: str, replacement: str, count: Optional[int] = None
) -> str:
    """Edits an existing UTF-8 file by replacing occurrences of a target string

    with new content.

    Args:
        path: Relative path of the file to edit.
        target: The exact string snippet to be replaced.
        replacement: The replacement string.
        count: Optional maximum number of occurrences to replace. Replaces all
          matches if omitted.

    Returns:
        A summary string reporting the number of replacements made.
    """
    file_path = resolve_work_path(path)
    try:
        original_content = file_path.read_text(encoding="utf-8")

        if target not in original_content:
            raise ValueError(f"Target string not found in {path}")

        occurrences = original_content.count(target)
        if count is not None and count > 0:
            new_content = original_content.replace(target, replacement, count)
            replaced_count = min(occurrences, count)
        else:
            new_content = original_content.replace(target, replacement)
            replaced_count = occurrences

        file_path.write_text(new_content, encoding="utf-8")
        return f"Successfully made {replaced_count} replacement(s) in {path}"
    except FileNotFoundError:
        raise FileNotFoundError("File does not exist")
    except PermissionError:
        raise PermissionError("Permission Error")
    except ValueError as e:
        raise ValueError(str(e))
    except Exception as e:
        raise Exception(f"Error editing the file: {e}")

In [22]:
@tool
def read_file_lines(path: str, start_line: int = 1, end_line: int = 100) -> str:
    """
    Reads a specific line range (1-indexed, inclusive) from a UTF-8 file.
    """
    file_path = resolve_work_path(path)
    try:
        with file_path.open("r", encoding="utf-8") as f:
            lines = [line for i, line in enumerate(f, start=1) if start_line <= i <= end_line]
        return "".join(lines)
    except FileNotFoundError:
        raise FileNotFoundError("File does not exist")
    except Exception as e:
        raise Exception(f"Error reading line range: {e}")

In [23]:
import re

@tool
def search_in_file(path: str, pattern: str, max_matches: int = 10) -> str:
    """
    Searches a file for a regex or substring pattern and returns matching lines with line numbers.
    """
    file_path = resolve_work_path(path)
    try:
        results = []
        regex = re.compile(pattern)
        with file_path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f, start=1):
                if regex.search(line):
                    results.append(f"L{idx}: {line.strip()}")
                    if len(results) >= max_matches:
                        break
        return "\n".join(results) if results else "No matches found."
    except Exception as e:
        raise Exception(f"Search failed: {e}")